# CVM DATA MINER
## 1. Libraries and constants

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import zipfile
import io
dfp_url = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/"

## 2. Functions

In [2]:
def find_zips(url):
    """Returns a dictionary years as keys and zip urls as values"""
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    tags = soup.find_all("a")
    zip_dict = {}
    for tag in tags:
        href = tag.get("href")
        if not href: #if href is empty                                   
            continue
        if href.endswith(".zip"):
            year = int(href.removesuffix(".zip")[-4:])
            link = urljoin(url, href)
            zip_dict[year] = link
    return zip_dict

def open_csv(zip_file, statement, basis, year):
    """Reads a CVM statement CSV from a DFP zip and returns it as a DataFrame"""
    with zipfile.ZipFile(zip_file) as z:
        csv_name = f"dfp_cia_aberta_{statement}_{basis}_{year}.csv"
        df = pd.read_csv(z.open(csv_name), sep=";", encoding="latin-1")
    return df

def download_zip(zip_link):
    """Downloads a zip from the given URL and returns it as an in-memory buffer"""
    response = requests.get(zip_link, timeout=60)
    response.raise_for_status()
    return io.BytesIO(response.content)

In [4]:
year = 2025
statement = "DFC_MI"
basis = "con"
zip_links = find_zips(dfp_url)
zip_data = download_zip(zip_links[year])
df = open_csv(zip_data, statement, basis, year)